# Feature Selection

The previous notebook used Ridge and Lasso to stop a model from leaning too hard on any one
feature. This notebook asks the question one step earlier: **which features should be in the
model at all?**

More features are not better. Every feature you add costs you something:

- **Overfitting.** Each extra feature gives the model another way to memorise noise.
- **Multicollinearity.** Features that carry the same information make coefficients unstable
  and impossible to interpret, which you saw on Monday when `horsepower`, `weight` and
  `displacement` turned out to be strongly correlated.
- **Cost.** Every feature has to be collected, cleaned and maintained for every future
  prediction.

There is no single correct method, so this notebook works through five, and ends with the
one question none of them answer on their own.

## Learning Objectives

At the end of this notebook, you should be able to:

- Rank features by their correlation and mutual information with the target, and list what correlation misses.
- Select the `k` highest-scoring features with `SelectKBest`, and compare the features two scoring functions keep.
- Detect multicollinearity with the variance inflation factor.
- Extract the p-values from a fitted model and identify the features whose effect is distinguishable from zero.
- Select features automatically with recursive feature elimination, sequential selection and Lasso.
- Choose the right association measure when a feature is categorical rather than numeric.
- Classify a selection method as filter, wrapper or embedded, and state whether it uses the target.

## Supervised and Unsupervised Feature Selection

Before looking at any particular method, two questions help to sort them. The first is the broadest one you can
ask of a selection method: **does it use the target?**

**Supervised** methods score a feature by its relationship with what you are predicting. They need
`y`, and the answer they give is specific to that target:

- **Correlation** with the target, and **mutual information**, which also catches non-linear
  relationships that correlation misses, plus **`SelectKBest`**, which keeps the top `k` features
  by either score
- **p-values** from a fitted model
- **RFE** and **Lasso**, which judge features through a fitted model, by the size of their coefficients, and
  **sequential selection**, which judges them by the model's score

**Unsupervised** methods never look at the target at all. They ask only whether a feature is worth
keeping on its own terms:

- A **variance threshold**, dropping columns that barely vary, plus constant and duplicate columns
- **VIF**, which asks whether a feature is redundant given the *other features*

Unsupervised screens usually come first, because a constant or duplicated column can go before you
even look at `y`, and they cost nothing to run.

### How Supervised Methods Relate to the Model: Filter, Wrapper, Embedded

Within the supervised group, the methods differ by how much they lean on a fitted model:

| Family | How it works | Cost | In this notebook |
| :--- | :--- | :--- | :--- |
| **Filter** | Scores each feature with a statistic, independently of any model. | Cheap | Correlation, mutual information, `SelectKBest`, VIF |
| **Wrapper** | Trains a model repeatedly on different subsets and keeps the best one. | Expensive | RFE |
| **Embedded** | Selection falls out of fitting the model, because the algorithm does it while training. | Free, once you fit | Lasso |

The boundaries are a guide rather than a strict partition. Reading p-values off a fitted model,
Method 3 below, sits between filter and embedded: the statistic is a filter-style score, but you
need a fitted model to get it. Note also that the two axes cross: VIF is a filter method that is
unsupervised, while correlation is a filter method that is supervised.

Both questions at a glance:

<img src="../assets/selection_method_families.png" width="700">

### What Feature Selection Is Not

> Feature selection keeps a **subset of the columns you started with**, which is what lets you say
> "we dropped `Population`". Dimensionality reduction instead builds **new** features by combining
> the originals, so the count falls but the original columns are no longer there to interpret.
> PCA does this without using the target (unsupervised); LDA does it using the class labels
> (supervised). Both are feature *extraction*, not selection, and they belong to a later session.

In [ ]:
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.feature_selection import (
    RFE,
    SelectKBest,
    SequentialFeatureSelector,
    VarianceThreshold,
    f_regression,
    mutual_info_regression,
)
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## The Data

The **California housing** dataset: one row per district, with the median house value as the
target. It is a housing regression problem, the same shape as the project at the end of this
week, but different data.

Everything here is numeric, which keeps the first five methods simple. The last section is
about what happens when that is not true.

Documentation: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
df = pd.read_csv("../data/california_housing.csv")

X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"{X_train.shape[0]} training rows, {X_train.shape[1]} features")
X_train.head()

Two of the methods below, RFE and Lasso, compare features by the **size of their coefficients**. As you saw in notebook 02, a coefficient's size depends on the units of its feature, so these two methods need **standardised** features. We fit the scaler on the training set only.
Scaling itself is covered in more depth tomorrow, in W3D3 (Feature Engineering).

Correlation, VIF and p-values do not depend on the units, so they use the original features.

Documentation: [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### A First Screen: Variance Threshold

*Filter method, unsupervised: looks only at the features.*

The cheapest check comes before any method that looks at the target: does each feature vary at all? A column that is constant, or almost constant, carries no information for any model. scikit-learn's [`VarianceThreshold`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html) removes every feature whose variance is at or below a threshold; with `threshold=0` it removes exactly the constant ones.

In [ ]:
selector_vt = VarianceThreshold(threshold=0).fit(X_train)
print(f"kept {selector_vt.get_support().sum()} of {X_train.shape[1]} features")

X_train.var().round(3)

All 8 features vary, so nothing is removed. That is the usual result on a curated dataset, but the check still pays off on raw data, where constant or duplicated columns are common.

Look at the variances, though: `Population` has a variance of about 1.3 million, `AveBedrms` of 0.193. A threshold above zero would judge features by their **units**, exactly like the unscaled penalty in notebook 02. In practice you use `threshold=0`, or a small threshold on features that share a scale, such as 0/1 columns.

> **Split first, then select.** Feature selection is a decision made *from* the data, so
> making it while the test set is visible leaks information and flatters your final score.
> Every selection decision below uses only `X_train` and `y_train`. The test set appears only
> afterwards, to check how well a smaller model performs.

---

## Method 1: Correlation

*Filter method, supervised: scores each feature against the target, no model required.*

The simplest question: does this feature move with the target?

**Pearson's correlation** runs from -1 to 1. Near zero means no *linear* relationship;
near either end means a strong one. Features that barely correlate with the target are
candidates for removal, and pairs of features that correlate strongly with *each other* are
carrying duplicate information.

In [ ]:
# Correlation of every feature with the target, strongest first
correlations = X_train.corrwith(y_train).sort_values(key=abs, ascending=False)
print(correlations.round(3).to_string())

In [ ]:
sns.heatmap(X_train.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between features");

`MedInc`, the median income of the district, is by far the strongest single predictor.
At the other end, `Population` is almost uncorrelated with house value.

Correlation plays two roles here. Against the **target**, as in the ranking above, it is a supervised filter. Between **features**, as in the heatmap above, it is unsupervised: it ignores the target and looks for redundancy.

The heatmap shows the other half of the story: `AveRooms` and `AveBedrms` are strongly
correlated with each other, which is unsurprising, and `Latitude` and `Longitude` are
strongly correlated too, because the state of California runs diagonally from north-west to south-east.

**What correlation will not tell you:**

- It only sees **linear** relationships. A perfect U-shaped relationship scores near zero.
- There is **no principled cutoff**. Is 0.1 too low? It depends.
- It looks at one feature at a time, so it cannot see that two weak features are strong
  *together*.
- **It cannot be computed sensibly for categorical features.** The last section of this
  notebook returns to this problem.

### A Non-Linear Alternative: Mutual Information

*Filter method, supervised.*

**Mutual information** answers the same question as correlation, "does this feature tell me something about the target?", without assuming a straight line. It is 0 when the feature and the target are independent, and it grows the more knowing the feature narrows down the target, whatever the shape of the relationship.

Documentation: [`mutual_info_regression`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_regression.html).

In [ ]:
mi = pd.Series(
    mutual_info_regression(X_train, y_train, random_state=42), index=X_train.columns
)

pd.DataFrame({"correlation": correlations, "mutual information": mi}).sort_values(
    "mutual information", ascending=False
).round(3)

The two rankings disagree, and the disagreement is the lesson. **`Longitude` comes first** by mutual information (0.386), level with `MedInc`, yet its correlation with house value is almost zero (-0.047, 6th of 8). Location matters a great deal for house prices, but not in a straight line: prices are high on the coast and low inland, so moving east does not steadily raise or lower them. Correlation cannot see that; mutual information can.

The price is that mutual information gives no direction (no sign), needs more data to estimate well, and still looks at one feature at a time.

### Automating the Filter: `SelectKBest`

*Filter method, supervised.*

Correlation and mutual information give you a ranking, but you still have to decide where to cut it. [`SelectKBest`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html) makes the cut for you: it scores every feature with a function you choose, keeps the `k` highest-scoring ones and drops the rest. It is a scikit-learn transformer with `fit` and `transform`, like the scaler, so it learns its choice from the training data only and can sit inside a `Pipeline`.

The scoring function decides what "best" means:

| `score_func` | Target | Measures |
| :-- | :-- | :-- |
| [`f_regression`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_regression.html) | numeric | a linear relationship (an F-test built on the correlation) |
| `mutual_info_regression` | numeric | a relationship of any shape |
| `f_classif` | categorical | a difference in group means (ANOVA) |
| `chi2` | categorical | an association, for non-negative features |

We keep 5 of the 8 features, the same number RFE keeps in Method 4, so the results can be compared.

In [ ]:
# Keep the 5 features with the highest F-score against the target
selector_kbest = SelectKBest(score_func=f_regression, k=5).fit(X_train, y_train)

# .scores_ holds one score per feature, in the column order of X_train
kbest_scores = pd.Series(selector_kbest.scores_, index=X_train.columns)
print(kbest_scores.sort_values(ascending=False).round(1).to_string())

# get_feature_names_out() returns the names of the kept columns
kept_kbest = selector_kbest.get_feature_names_out()
print("\nkept:", list(kept_kbest))

The F-scores rank the features in exactly the same order as the absolute correlations above, because `f_regression` is computed from the correlation: it turns each correlation into a test statistic. `MedInc` scores about 14,000, far ahead of every other feature.

Now look at the cut. `AveBedrms` (34.9) is kept and `Longitude` (34.7) is dropped, by a margin of 0.2. Does that matter?

In [ ]:
small_kbest = LinearRegression().fit(X_train[kept_kbest], y_train)
rmse_kbest = root_mean_squared_error(y_test, small_kbest.predict(X_test[kept_kbest]))

print(f"SelectKBest with f_regression: test RMSE {rmse_kbest:.4f}")

It does. The test RMSE is **0.7903**, clearly worse than the 0.7356 of all eight features (computed in Method 4). The model kept `Latitude` but lost `Longitude`, and a location needs both: latitude alone tells you how far north a district lies, not whether it is on the coast or inland. A filter scores **one feature at a time**, so it cannot see that two features are useful mainly together.

The scoring function is a choice you can change. Here is the same selector with mutual information, which found the non-linear signal in `Longitude`:

In [ ]:
# mutual_info_regression involves randomness. partial() creates a copy of the
# function with random_state already filled in, so the selection is reproducible.
mi_score = partial(mutual_info_regression, random_state=42)
selector_mi = SelectKBest(score_func=mi_score, k=5).fit(X_train, y_train)
kept_mi = selector_mi.get_feature_names_out()

small_mi = LinearRegression().fit(X_train[kept_mi], y_train)
rmse_mi = root_mean_squared_error(y_test, small_mi.predict(X_test[kept_mi]))

print("kept:", list(kept_mi))
print(f"SelectKBest with mutual information: test RMSE {rmse_mi:.4f}")

Now both coordinates are in, `HouseAge` and `AveBedrms` are out, and the test RMSE drops to **0.7411**, close to the 0.7356 of all eight features with three features fewer.

Same selector, same `k`, a different score, and a different model. **The scoring function is the real decision**, and `k` is a second one that `SelectKBest` leaves to you. On Thursday you tune hyperparameters with cross-validation, and `k` can be tuned in exactly the same way.

---

## Method 2: Variance Inflation Factor

*Filter method, unsupervised: looks only at the features, never at the target.*

Correlation compares features in pairs. **Multicollinearity** can hide between three or more
features at once: no pair looks alarming, yet one feature is predictable from a combination
of the others.

The **variance inflation factor** catches this. For each feature, it fits a regression
predicting that feature from all the *other* features, and reports

$$VIF = \frac{1}{1 - R^2}$$

So a feature the others can reconstruct perfectly has an $R^2$ near 1 and a VIF heading for
infinity. The usual rough readings are **VIF above 5** is worth a look and **above 10** is a
problem.

Documentation: [`variance_inflation_factor`](https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html), [`add_constant`](https://www.statsmodels.org/stable/generated/statsmodels.tools.tools.add_constant.html).

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.api import add_constant

# add_constant appends a column of ones, which VIF needs as the regression intercept.
# variance_inflation_factor is one of the few functions that needs a NumPy array (.values).
X_vif = add_constant(X_train)

vif = pd.Series(
    [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
    index=X_vif.columns,
    name="VIF",
)

print(vif.drop("const").sort_values(ascending=False).round(2).to_string())

`AveRooms` and `AveBedrms` both score high, confirming what the heatmap suggested: they say
much the same thing. Dropping one of the pair usually costs almost no predictive power and
makes the surviving coefficient interpretable again.

Note that VIF says nothing about the target. It is purely a statement about redundancy
*among the features*.

---

## Method 3: Significance, with p-values

*Between filter and embedded: a statistic, but one you can only read off a fitted model.*

A third angle: fit the model, then ask whether each coefficient is distinguishable from zero.

The **p-value** for a coefficient answers: if this feature truly had no effect, how often
would we see a coefficient at least this large by chance? A small p-value, conventionally
below 0.05, means the feature is unlikely to be pure noise.

scikit-learn does not report p-values, because it is built for prediction rather than
inference. `statsmodels` does. We need one number from it here, so there is no need to read
the whole summary table.

Documentation: [`OLS`](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).

In [ ]:
from statsmodels.api import OLS

# statsmodels does not add an intercept for you, hence add_constant again
model = OLS(y_train, add_constant(X_train)).fit()

p_values = model.pvalues.drop("const").sort_values()

print(p_values.round(4).to_string())

Most features here are significant, which is common with 15,000 rows: **large samples make
small effects statistically significant**. A p-value tells you the effect is probably not
zero. It does not tell you the effect is *large enough to care about*, and with enough data
almost nothing is exactly zero.

So read the p-value next to the coefficient, never instead of it.

---

## Method 4: Recursive Feature Elimination

*Wrapper method: refits the model on shrinking subsets, so it is more expensive than the filter methods.*

The three methods so far score features and leave the decision to you. **RFE** makes the
decision: fit the model, throw away the weakest feature, refit, and repeat until only the
requested number remain.

<img src="../assets/rfe_loop.png" width="350">

Because it refits after every removal, it can notice that a feature only looked weak in the
company it was keeping.

RFE decides which feature is weakest by the size of its coefficient, so we fit it on the
**scaled** training data.

Documentation: [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [`RFE`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html).

In [ ]:
selector = RFE(LinearRegression(), n_features_to_select=5)
selector.fit(X_train_scaled, y_train)

kept = X_train.columns[selector.support_]
dropped = X_train.columns[~selector.support_]

print("kept:   ", list(kept))
print("dropped:", list(dropped))

### Did Dropping Three Features Hurt?

The honest test of any selection method is whether the smaller model actually performs
worse.

Documentation: [`root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html).

In [ ]:
full = LinearRegression().fit(X_train, y_train)
small = LinearRegression().fit(X_train[kept], y_train)

rmse_full = root_mean_squared_error(y_test, full.predict(X_test))
rmse_small = root_mean_squared_error(y_test, small.predict(X_test[kept]))

print(f"all {X_train.shape[1]} features: RMSE {rmse_full:.4f}")
print(f"{len(kept)} features:      RMSE {rmse_small:.4f}")

Dropping three of eight features made the model slightly **worse**, not better: the RMSE
rises from 0.7356 to 0.7445. That is the usual result, and it is worth
thinking about. Those three features were carrying a little signal, so removing them costs a
little accuracy.

The trade is not accuracy against nothing. It is a tiny amount of accuracy against a model
that is smaller, cheaper to maintain, easier to explain and less prone to overfitting on
data it has not seen. **Whether that trade is worth making is a judgement, not a
calculation**, and it is yours to make rather than the method's.

### Try It: Forward and Backward Selection

RFE is one wrapper method. Two more are **forward selection**, which starts with no features and adds the most useful one at each step, and **backward elimination**, which starts with all of them and removes the least useful one. Unlike RFE, they judge each step by the model's **score**, not by the size of its coefficients, so they do not need scaled features.

scikit-learn offers both in [`SequentialFeatureSelector`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html), set with `direction="forward"` or `direction="backward"`. Keep 5 features with each direction, compare the features they keep with RFE's, and compare their test RMSE with RFE's 0.7445.

In [ ]:
for direction in ["forward", "backward"]:
    sfs = SequentialFeatureSelector(
        LinearRegression(), n_features_to_select=5, direction=direction
    )
    sfs.fit(X_train, y_train)
    kept_sfs = X_train.columns[sfs.get_support()]

    small_sfs = LinearRegression().fit(X_train[kept_sfs], y_train)
    rmse_sfs = root_mean_squared_error(y_test, small_sfs.predict(X_test[kept_sfs]))
    print(f"{direction:8}: {list(kept_sfs)}  test RMSE {rmse_sfs:.4f}")

<details><summary>Show answer</summary>

```python
for direction in ["forward", "backward"]:
    sfs = SequentialFeatureSelector(
        LinearRegression(), n_features_to_select=5, direction=direction
    )
    sfs.fit(X_train, y_train)
    kept_sfs = X_train.columns[sfs.get_support()]

    small_sfs = LinearRegression().fit(X_train[kept_sfs], y_train)
    rmse_sfs = root_mean_squared_error(y_test, small_sfs.predict(X_test[kept_sfs]))
    print(f"{direction:8}: {list(kept_sfs)}  test RMSE {rmse_sfs:.4f}")
```

Both directions keep the same five features: `MedInc`, `HouseAge`, `AveBedrms`, `Latitude` and `Longitude`. The test RMSE is **0.7339**: better than RFE's five (0.7445), and even slightly better than all eight features (0.7356).

The difference from RFE is a single feature: RFE kept `AveRooms`, sequential selection keeps `HouseAge`. RFE drops whichever feature has the smallest coefficient, so it kept `AveRooms`, which has a large coefficient because it works against its correlated partner `AveBedrms`. Sequential selection asks which feature actually improves the score, and `HouseAge` does.

The price is time: sequential selection fits many models, and scores each candidate with cross-validation (by default 5 folds), which you will learn in W3D4.

</details>

---

## Method 5: Lasso

*Embedded method: selection is a side effect of the penalised fit.*

You met Lasso in the previous notebook as a way of shrinking coefficients. Shrinking is
exactly what makes it a selection method: **Lasso drives some coefficients to exactly zero**,
and a feature with a zero coefficient has been removed from the model.

This is selection as a side effect of fitting, which is the appeal. The strength of the
penalty, `alpha`, decides how aggressive it is.

Documentation: [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html).

In [ ]:
lasso = Lasso(alpha=0.05).fit(X_train_scaled, y_train)

coefficients = pd.Series(lasso.coef_, index=X_train.columns)

zeroed = coefficients[coefficients == 0].index

print("Lasso set these to exactly zero:", list(zeroed))
print()
print(coefficients.round(3).to_string())

At `alpha = 0.05`, Lasso keeps four features, `MedInc`, `HouseAge`, `Latitude` and `Longitude`,
and sets the other four exactly to zero. Only an exact zero means the feature has left the
model. Ridge shrinks every coefficient towards zero without ever reaching it; Lasso can set it exactly to zero.

Now compare with RFE. Both keep `MedInc`, `Latitude` and `Longitude`, but they **disagree on
`AveRooms` and `AveBedrms`**: RFE kept both, Lasso dropped both. That is no accident. These two
are the correlated pair with high VIFs (8.12 and 6.80): each one carries much the same
information as the other, so a method can keep both or neither and lose little. Where the
methods disagree is exactly where the data deserves a closer look.

> **Scale matters for Lasso.** Without scaling, `Population` (measured in people, in the
> thousands) gets a coefficient of about 0.000006 only because of its units, and looks almost
> irrelevant. The penalty would judge features by their units, not their importance. That is
> why we fitted Lasso on the scaled features.

---

## The Question None of These Answered

Every method above needs a **number**. Correlation, mutual information, VIF, p-values, RFE and Lasso all assume
a feature you can multiply by a coefficient.

So what do you do with a feature like `neighbourhood`, or `has_waterfront`, or `condition`?

This is not a rare case. Most real datasets contain such features, and so does the project at the end of
this week: **King County stores `waterfront`, `view`, `condition`, `grade` and `zipcode` as
integers.** They will appear in your correlation matrix without any warning, and each one will be given a
coefficient. For `zipcode`, seventy arbitrary zip codes, that number is meaningless, and a
correlation heatmap will not warn you.

### The Tests You Already Have

You met the answer in week 2. The types of the feature and the target decide which test to use:

| Feature | Target | Measure |
| :------ | :----- | :------ |
| numeric | numeric | Pearson or Spearman correlation |
| **categorical** | **numeric** | **t-test** for two groups, **ANOVA** for three or more |
| **categorical** | **categorical** | **chi-square**, with **Cramer's V** as the effect size |

We use the `tips` data from that week to show it, because you have met it before.

In [ ]:
tips = pd.read_csv("../data/tips.csv")
tips.head()

### Categorical Feature Against a Numeric Target: ANOVA

**Question:** is `day` worth keeping as a feature for predicting `total_bill`?

Correlation cannot answer this. ANOVA can: it asks whether the group means differ by more
than sampling noise would explain.

Documentation: [`f_oneway`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html).

In [ ]:
# One array of total_bill values per day
groups = [group["total_bill"] for _, group in tips.groupby("day", observed=True)]

f_stat, p_value = stats.f_oneway(*groups)

print(f"ANOVA F = {f_stat:.3f}, p = {p_value:.4f}")
print(tips.groupby("day", observed=True)["total_bill"].mean().round(2).to_string())

The group means differ by a few dollars and the p-value lands at 0.043, just under the
conventional 0.05. **This is the same borderline result week 2 used to introduce the
multiple-testing trap**, and it should be read the same way here: a p-value that close to
the line, found among several tests, is a candidate for a second look rather than a finding.

Borrowing a test from EDA also borrows its cautions.

### Categorical Feature Against a Categorical Target: Cramer's V

**Question:** do `smoker` and `time` carry the same information? If they did, keeping both
would be the categorical version of multicollinearity.

Chi-square tests whether the two are associated. Cramer's V turns the test statistic into an
effect size between 0 and 1, which is the part you compare across pairs.

The same chi-square test is also the standard filter when the **target** is categorical, as in
classification: scikit-learn offers it as `SelectKBest(chi2)`, the same selector you used with `f_regression`. With a numeric target like house
value it does not apply.

Documentation: [`chi2_contingency`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html).

In [ ]:
contingency = pd.crosstab(tips["smoker"], tips["time"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

# Cramer's V: effect size for a contingency table, from 0 (none) to 1 (perfect)
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print(contingency.to_string())
print(f"\nchi-square p = {p_value:.4f}, Cramer's V = {cramers_v:.3f}")

A Cramer's V near zero means the two features are independent, so each is telling you
something the other is not and both can stay.

---

## Which Method, When?

| Method | Family | Answers | Watch out for |
| :----- | :----- | :------ | :------------ |
| Variance threshold | Filter, unsupervised | Does this feature vary at all? | Any threshold above 0 depends on the units |
| Correlation | Filter, supervised (with the target) or unsupervised (between features) | Does this feature move with the target, or with another feature? | Linear only, numeric only, no cutoff |
| Mutual information | Filter, supervised | Does this feature tell us about the target, in any shape? | No sign, needs enough data, one feature at a time |
| `SelectKBest` | Filter, supervised | Which `k` features score highest? | The result depends on the scoring function, you have to choose `k`, and features useful only together can be split |
| VIF | Filter, unsupervised | Is this feature redundant given the others? | Says nothing about the target |
| p-values | Filter / embedded | Is this effect distinguishable from zero? | Large samples make everything significant |
| RFE | Wrapper | Which n features work best together? | Requires scaling, you have to choose `n`, and it refits repeatedly |
| Forward and backward selection | Wrapper | Which feature improves the score most at each step? | Fits many models, scored with cross-validation |
| Lasso | Embedded | Which features survive a penalty? | Requires scaling, and `alpha` has to be tuned |
| ANOVA, Cramer's V | Filter | Does this **categorical** feature matter? | The only option once a feature is not numeric |
| Tree-based importance | Embedded | Which features do a tree's splits rely on? | Comes with decision trees and random forests in week 4 |

**In practice you use several.** They disagree, and where they disagree is where you learn
something about the data.

## Summary

In this notebook you:

- Screened for constant features with a variance threshold.
- Ranked features by their correlation with the target, and saw that it only catches linear relationships, while mutual information also found the non-linear signal in `Longitude`.
- Kept the top 5 features with `SelectKBest`, and saw that switching the score from `f_regression` to mutual information changed the selection and the test RMSE (0.7903 against 0.7411).
- Detected multicollinearity with the variance inflation factor.
- Extracted p-values from a fitted model, and saw that a large sample makes even small effects significant.
- Selected features with RFE, sequential selection and Lasso, and checked whether the smaller model still performs.
- Chose ANOVA and Cramer's V for categorical features.
- Classified the methods as filter, wrapper or embedded, and as supervised or unsupervised.

Keep in mind:

- Feature selection happens **after the train/test split**, using training data only.
- Methods group into **filter** (a statistic, no model), **wrapper** (refit on subsets) and
  **embedded** (selection falls out of training), and they are not all supervised: VIF and a
  variance threshold never look at the target.
- Selection keeps original columns. **PCA and LDA build new ones**, so they reduce dimensions
  without selecting features.
- **Correlation and VIF** are quick screens: one against the target, one for redundancy.
- **p-values, RFE and Lasso** all use a fitted model, and Lasso gets selection for free.
- A number from any of these is evidence, not a verdict. The honest test is whether the
  smaller model still performs.
- **Numeric methods mishandle categorical features silently.** When a feature is a category,
  reach for week 2's tests, and remember that a category stored as an integer looks numeric
  to every tool in this notebook.

## References & Further Reading

### The scikit-learn Feature Selection Guide

The [**scikit-learn feature selection user guide**](https://scikit-learn.org/stable/modules/feature_selection.html) has one section for every family of selectors the library offers. This table maps each section to where it appears in this notebook:

| Section of the guide | What it covers | In this notebook |
| :-- | :-- | :-- |
| [Removing features with low variance](https://scikit-learn.org/stable/modules/feature_selection.html#removing-features-with-low-variance) | `VarianceThreshold` | A First Screen: Variance Threshold |
| [Univariate feature selection](https://scikit-learn.org/stable/modules/feature_selection.html#univariate-feature-selection) | `SelectKBest` and the other univariate selectors, with the scoring functions `f_regression`, `r_regression`, `mutual_info_regression` (numeric target) and `f_classif`, `chi2`, `mutual_info_classif` (categorical target) | Mutual information and `SelectKBest` |
| [Recursive feature elimination](https://scikit-learn.org/stable/modules/feature_selection.html#recursive-feature-elimination) | `RFE`, and `RFECV`, which chooses the number of features with cross-validation | Method 4 |
| [Sequential feature selection](https://scikit-learn.org/stable/modules/feature_selection.html#sequential-feature-selection) | `SequentialFeatureSelector`, forward and backward | Try It: Forward and Backward Selection |
| [Feature selection using SelectFromModel](https://scikit-learn.org/stable/modules/feature_selection.html#feature-selection-using-selectfrommodel) | `SelectFromModel`, which keeps the features whose coefficient or importance passes a threshold | Not used here |
| [L1-based feature selection](https://scikit-learn.org/stable/modules/feature_selection.html#l1-based-feature-selection) | `SelectFromModel` around Lasso, which automates the zero-coefficient check | Method 5, done by hand |
| [Tree-based feature selection](https://scikit-learn.org/stable/modules/feature_selection.html#tree-based-feature-selection) | `SelectFromModel` around a random forest, using its feature importances | Week 4 (decision trees and random forests) |
| [Feature selection as part of a pipeline](https://scikit-learn.org/stable/modules/feature_selection.html#feature-selection-as-part-of-a-pipeline) | A selector as one step of a `Pipeline`, so it is refitted inside every cross-validation fold | Thursday (cross-validation and hyperparameter tuning) |

### scikit-learn Selectors Not Used in This Notebook

- [**SelectPercentile**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectPercentile.html): like `SelectKBest`, but keeps a percentage of the features instead of a fixed number.
- [**SelectFpr**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFpr.html), [**SelectFdr**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFdr.html) and [**SelectFwe**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFwe.html): keep the features whose p-value passes a threshold. `SelectFdr` and `SelectFwe` correct for testing many features at once, the multiple-testing problem from week 2.
- [**GenericUnivariateSelect**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.GenericUnivariateSelect.html): all the univariate selectors in one class, with the strategy as a parameter, so the strategy itself can be tuned.
- [**RFECV**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html): RFE that picks the number of features by cross-validation instead of asking you for it.
- [**SelectFromModel**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html): turns any fitted model with coefficients or feature importances into a selector, for example Lasso or a random forest.
- [**The full `sklearn.feature_selection` API**](https://scikit-learn.org/stable/api/sklearn.feature_selection.html): every class and scoring function in the module.

### Classes and Functions Used in This Notebook

- [**SelectKBest (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html): the univariate selector that keeps the `k` best features by a scoring function of your choice.
- [**RFE (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html): the recursive feature elimination class used in Method 4.
- [**SequentialFeatureSelector (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html): forward and backward selection, used in the Try It after Method 4.
- [**Variance inflation factor (statsmodels)**](https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html): the function used in Method 2.
- [**One-way ANOVA (SciPy)**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html): the test used for a categorical feature against a numeric target.